# Import libraries

In [1]:
# Changes to all modules will automatically be applied when any cell runs. 
%load_ext autoreload
%autoreload 2

In [2]:
import logging

# Set logging level for PyTorch Tabular
logging.getLogger("pytorch_tabular").setLevel(logging.ERROR)

# Set logging level for PyTorch Lightning
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

# Set logging level for Lightning Fabric
logging.getLogger("lightning_fabric").setLevel(logging.ERROR)

# Disable device availability messages
logging.getLogger("lightning.pytorch.utilities.rank_zero").setLevel(logging.FATAL)

import pandas as pd
import numpy as np

from pytorch_tabular.models import CategoryEmbeddingModelConfig, TabNetModelConfig

import optuna

from functools import partial

from pathlib import Path
import sys

from typing import Type
from sklearn.base import BaseEstimator

from sklearn.metrics import (
    make_scorer,
    f1_score,
)

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from split_tools import get_train_test
from modelling4_utils import (
    # _prepare_df,
    MLPipeline,
    StatsModelsEstimator,
    PytorchTabularEstimator,
    deep_update_estimator_params,
    OptunaOptimizer,
    GridSearchOptimizer,
    RANDOM_STATE,
)

# from pytorch_lightning.loggers import MLFlowLogger
from pytorch_lightning.loggers import TensorBoardLogger

seed = RANDOM_STATE

# Fix error with save weights

# import torch
# # from omegaconf import DictConfig
# # from omegaconf.base import ContainerMetadata
# # import typing
# # torch.serialization.add_safe_globals([DictConfig, ContainerMetadata, typing.Any])
# import pytorch_tabular.utils.python_utils as pt_utils
# pt_utils.pl_load = lambda f, map_location: torch.load(f, map_location=map_location, weights_only=False)



In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings("ignore")

In [4]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())

True
1


# Settings

In [5]:
model_postfix = 'narrow_opt-model'

scoring_metrics = {"f1_macro": make_scorer(f1_score, average="macro"),}
step_scoring_average = "mean"
n_trials = 300 # make 500

save_model_and_metrics = True
metrics_file = "metrics_modelling4_6-neural_networks.xlsx"

## Optimization function

In [6]:
def optimize_estimator_optuna(
    target:str,
    estimator:Type[BaseEstimator],
    estimator_params:dict,
    objective:callable,
    n_trials:int,
    add_smote:bool,
    is_smotenc:bool,
    smote_params:dict,
    step_scoring_average:str=step_scoring_average,
    params_to_process:list=None,
    model_postfix:str=model_postfix,
    scoring_metrics:dict=scoring_metrics,
    save_model_and_metrics:bool=save_model_and_metrics,
    metrics_file:str=metrics_file,
    opt_cv_folds:int=5,
    seed:int=seed,
):
    
    estimator_params = estimator_params.copy()
    
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
        scoring_metrics=scoring_metrics,
        step_scoring_average=step_scoring_average,
        cv_folds=opt_cv_folds,
    )
    
    opt = OptunaOptimizer(
        objective=partial(objective, ml_pipe=ml_pipe),
        study_name="model_study",
        direction="maximize",
        seed=seed,
    )
    
    opt.optimize(n_trials=n_trials)
    
    best_params = opt.study.best_params
    
    print('raw best_params')
    display(best_params)
    
    if params_to_process:
        for param in params_to_process:
            if param == 'log2_size':
                keys = [k for k in best_params.keys() if param in k]
                sorted_keys = sorted(keys, key=lambda k: int(k.split("_")[1]))
                layer_sizes = [
                    2**best_params.pop(key)
                    for key in sorted_keys
                ]
                best_params['layers'] = '-'.join(map(str, layer_sizes))
            elif param == 'num_layers':
                key = next((k for k in best_params.keys() if param in k), None)
                if key:
                    best_params.pop(key)
            else: # Other cases
                # Find one key in best_params which contains param
                key = next((k for k in best_params.keys() if param in k), None)
                if key:
                    best_params[param] = best_params.pop(key)

    print('best_params')
    display(best_params)
    
    suggested_estimator_params = {
        "model_config_params": best_params
    }
    best_estimator_params = deep_update_estimator_params(
        ml_pipe=ml_pipe,
        suggested_params=suggested_estimator_params,
    )
    
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params={**estimator_params, **best_estimator_params},
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
        step_scoring_average=step_scoring_average, # No need to pass, it would not be used
        metrics_file=metrics_file,
    )
    
    ml_pipe.run(
        save_model_and_metrics=save_model_and_metrics,
    )

## TabNet

In [7]:

def TabNet_objective(
    trial:optuna.trial.Trial, 
    ml_pipe:MLPipeline,
):
    # decoder dimension
    n_d = trial.suggest_int('n_d', 4, 16, step=4)
    # attention dimension
    n_a = trial.suggest_int('n_a', 4, 16, step=4)
    # number of steps of the attention mechanism
    n_steps = trial.suggest_int('n_steps', 2, 4)
    # sparsity parameter - controls the importance of the sparsity loss
    gamma = trial.suggest_float('gamma', 1.0, 1.8)
    # number of shared GLU layers (shared across steps & features)
    n_shared = trial.suggest_int('n_shared', 1, 2)
    # number of independent GLU layers (shared across decision steps)
    n_independent = trial.suggest_int('n_independent', 1, 2)
    # learning rate
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    
    suggested_estimator_params = {
        "model_config_params": {
            'n_d': n_d,
            'n_a': n_a,
            'n_steps': n_steps,
            'gamma': gamma,
            'n_shared': n_shared,
            'n_independent': n_independent,
            'learning_rate': learning_rate,
        }
    }
    
    estimator_params = deep_update_estimator_params(
        ml_pipe=ml_pipe,
        suggested_params=suggested_estimator_params,
    )
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        estimator_params=estimator_params,
    )
    
    return score


In [8]:
estimator = PytorchTabularEstimator
target = "splashing"

add_smote = True
is_smotenc = False
smote_params = {
    # 'categorical_features': (
    #     'wettability',
    #     'inclination',
    # ),
}

estimator_params = {
    "model_class": TabNetModelConfig,
    # Other params SET in optuna
    "model_config_params": {
        'virtual_batch_size': 16,
    },
    # "data_config_params": {},
    "trainer_config_params": {
        'max_epochs': 200,
        'batch_size': 64, # let's condiser whole dataset
        'early_stopping': 'valid_loss',
        'early_stopping_patience': 10,
        'accelerator': 'gpu',
        'devices': 1,
        'progress_bar': 'none',
    },
    "optimizer_config_params": {
        'optimizer': 'Adam',
        'lr_scheduler': 'ReduceLROnPlateau',
        'lr_scheduler_params': {
            'patience': 5,
            'factor': 0.5,
        },
    },
}

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    objective=TabNet_objective,
    n_trials=n_trials,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-29 13:07:43,120] A new study created in memory with name: model_study
[I 2025-04-29 13:08:06,387] Trial 0 finished with value: 0.6397549614924141 and parameters: {'n_d': 8, 'n_a': 16, 'n_steps': 4, 'gamma': 1.4789267873576293, 'n_shared': 1, 'n_independent': 1, 'learning_rate': 0.00013066739238053285}. Best is trial 0 with value: 0.6397549614924141.
[I 2025-04-29 13:09:59,143] Trial 1 finished with value: 0.6033739386759807 and parameters: {'n_d': 16, 'n_a': 12, 'n_steps': 4, 'gamma': 1.016467595436642, 'n_shared': 2, 'n_independent': 2, 'learning_rate': 0.00026587543983272726}. Best is trial 0 with value: 0.6397549614924141.
[I 2025-04-29 13:11:44,669] Trial 2 finished with value: 0.7623922418518341 and parameters: {'n_d': 4, 'n_a': 4, 'n_steps': 2, 'gamma': 1.4198051453057903, 'n_shared': 1, 'n_independent': 1, 'learning_rate': 0.0016738085788752138}. Best is trial 2 with value: 0.7623922418518341.
[I 2025-04-29 13:13:41,865] Trial 3 finished with value: 0.2849074074074074

raw best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "PytorchTabularEstimator"


,0
target,splashing
model,TabNetModel_splashing_smote_narrow_opt-model
holdout_test_f1_macro,0.829027
holdout_test_accuracy_balanced,0.834491
holdout_test_roc_auc,0.911265
holdout_test_f1,0.87234
holdout_test_accuracy,0.84
cv_test_f1_macro_median,0.835913
cv_test_accuracy_balanced_median,0.835913
cv_test_roc_auc_median,0.911765


Model saved in ..\results\models_modelling4\TabNetModel_splashing_smote_narrow_opt-model
